In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find MedGemma-27b-text-it project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"

# Full path to the model snapshot
model_path = "/orcd/compute/mghassem/001/gobi1/huggingface/hub/models--google--medgemma-27b-text-it/snapshots/5b667cf2ddcf064085bc90952edb35a0edbfb79c"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)

prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

/home/yuexing/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 808/808 [01:20<00:00, 10.03it/s, Materializing param=model.norm.wei
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Okay, here's a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to understand, generate, and interact with human language.**

Think of them as incredibly sophisticated pattern-matching machines. They are trained on massive amounts of text data (like books, articles, websites) and learn the statistical relationships between words and concepts.

**Key characteristics:**

*   **"Large":** They have billions (or even trillions) of parameters, which are essentially the variables the model adjusts during training to learn patterns.
*   **"Language":** Their primary function is processing and generating human language.
*   **Capabilities:** They can perform tasks like:
    *   Answering questions
    *   Writing essays, code, or creative content
    *   Translating languages
    *   Summarizing text
    *   Holding conversations (like chatbots)

**In essence, LLMs are powerful tools that can mimic human-lik

In [ ]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None

# Check if progress file exists and load it
progress_file = paths.PREDICTIONS / "[SR]_MedGemma27B_predictions_on_14B_progress.csv"
output_file = "[SR]_MedGemma27B_predictions_14B.csv"

if os.path.exists(progress_file):
    print(f"Found existing progress file: {progress_file}")
    existing_df = pd.read_csv(progress_file)
    results = existing_df.to_dict('records')
    processed_indices = set(range(len(results)))
    start_idx = len(results)
    print(f"Resuming from row {start_idx + 1} (already processed {start_idx} rows)")
else:
    print("No existing progress file found. Starting from scratch.")
    results = []
    processed_indices = set()
    start_idx = 0

# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx in range(start_idx, total_rows):
    row = df.iloc[idx]
    print(f"Processing row {idx+1}/{total_rows}...")
    
    try:
        context_text = row["Qwen14B_High_Relevance"]
        question = row["question_options_x"]
        
        # Improved prompt with clearer instructions
        query_full = (
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Based on the information provided, select the correct answer choice (A, B, C, D, etc.).\n\n"
            "IMPORTANT: Your response must end with 'Answer: X' where X is the letter of your chosen option.\n"
            "For example: Answer: A"
        )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""), 
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"💾 Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        
        # Save progress even on error
        temp_df = pd.DataFrame(results)
        temp_df.to_csv(progress_file, index=False)
        print(f"💾 Saved progress after error on row {idx+1}")

# Save final results
output_df = pd.DataFrame(results)
output_df.to_csv(output_file, index=False)
print(f"✅ Saved all predictions to {output_file}")
print(f"Total processed: {len(results)}/{total_rows}")

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21', 'Qwen14B_answer', 'Qwen14B_raw_response', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21', 'Qwen14B_High_Relevance']
Found existing progress file: [SR]_MedGemma27B_predictions_on_14B_progress.csv
Resuming from row 471 (already processed 470 rows)
Processing 1303 rows...
Processing row 471/1303...
⚠️ Could not extract answer from response for ro

✅ Processed Merge Q506: Answer = B
Processing row 507/1303...
✅ Processed Merge Q507: Answer = C
Processing row 508/1303...
⚠️ Could not extract answer from response for row 508:
Response: The patient has a history of a pulmonary abscess and subsequent scar tissue formation in the left lo...
✅ Processed Merge Q508: Answer = None
Processing row 509/1303...
⚠️ Could not extract answer from response for row 509:
Response: **Analysis of the Case:**

1.  **Patient Profile:** 17-year-old female.
2.  **Presenting Symptoms:**...
✅ Processed Merge Q509: Answer = None
Processing row 510/1303...
⚠️ Could not extract answer from response for row 510:
Response: **Patient Information:**
*   **Age:** 55 years old
*   **Gender:** Male
*   **Chief Complaint:** Rou...
✅ Processed Merge Q510: Answer = None
💾 Saved progress to CSV after 510 items
Processing row 511/1303...
✅ Processed Merge Q511: Answer = A
Processing row 512/1303...
⚠️ Could not extract answer from response for row 512:
Response: **Analy

⚠️ Could not extract answer from response for row 550:
Response: **Analysis of the provided information:**

1.  **Patient Presentation:** A 65-year-old man presents ...
✅ Processed Merge Q550: Answer = None
💾 Saved progress to CSV after 550 items
Processing row 551/1303...
⚠️ Could not extract answer from response for row 551:
Response: **Analysis:**

1.  **Identify the key symptoms:** The patient presents with postprandial right upper...
✅ Processed Merge Q551: Answer = None
Processing row 552/1303...
⚠️ Could not extract answer from response for row 552:
Response: **Analysis of the Case:**

The patient presents with classic signs and symptoms of a carotid caverno...
✅ Processed Merge Q552: Answer = None
Processing row 553/1303...
⚠️ Could not extract answer from response for row 553:
Response: **Analysis of the Case:**

1.  **Diagnosis:** The patient has severe, treatment-refractory Obsessive...
✅ Processed Merge Q553: Answer = None
Processing row 554/1303...
⚠️ Could not extract ans

⚠️ Could not extract answer from response for row 588:
Response: **Analysis:**

1.  **Identify the patient's condition:** The patient is a 28-year-old male involved ...
✅ Processed Merge Q588: Answer = None
Processing row 589/1303...
✅ Processed Merge Q589: Answer = A
Processing row 590/1303...
⚠️ Could not extract answer from response for row 590:
Response: **Analysis of Findings:**

1.  **Seated Flexion Test:** The left posterior superior iliac spine (PSI...
✅ Processed Merge Q590: Answer = None
💾 Saved progress to CSV after 590 items
Processing row 591/1303...
⚠️ Could not extract answer from response for row 591:
Response: **Analysis of the Case:**

*   **Patient:** 58-year-old female.
*   **Presenting Complaint:** Ptosis...
✅ Processed Merge Q591: Answer = None
Processing row 592/1303...
⚠️ Could not extract answer from response for row 592:
Response: **Analysis of the Case:**

1.  **Symptoms:** The neonate exhibits cyanosis (dusky blue discoloration...
✅ Processed Merge Q592: Ans

⚠️ Could not extract answer from response for row 632:
Response: The patient is a 30-year-old man presenting with a 7-year history of involuntary orofaciolingual mov...
✅ Processed Merge Q632: Answer = None
Processing row 633/1303...
✅ Processed Merge Q633: Answer = C
Processing row 634/1303...
⚠️ Could not extract answer from response for row 634:
Response: **Analysis of the Case:**

*   **Patient:** 42-year-old male.
*   **Symptoms:** Abdominal pain, weig...
✅ Processed Merge Q634: Answer = None
Processing row 635/1303...
⚠️ Could not extract answer from response for row 635:
Response: **Analysis:**

1.  **Identify the core problem:** The patient presents with symptoms (dyspnea, cough...
✅ Processed Merge Q635: Answer = None
Processing row 636/1303...
⚠️ Could not extract answer from response for row 636:
Response: **Analysis of the provided information:**

1.  **Diagnosis:** Chronic Lymphocytic Leukemia (CLL).
2....
✅ Processed Merge Q636: Answer = None
Processing row 637/1303...
✅ 

⚠️ Could not extract answer from response for row 676:
Response: **Analysis:**

1.  **Identify the key clinical findings:**
    *   Unilateral (right eye) vision los...
✅ Processed Merge Q676: Answer = None
Processing row 677/1303...
⚠️ Could not extract answer from response for row 677:
Response: **Analysis of the Case:**

1.  **Patient Profile:** 3-week-old infant (neonate/young infant).
2.  **...
✅ Processed Merge Q677: Answer = None
Processing row 678/1303...
⚠️ Could not extract answer from response for row 678:
Response: **Analysis:**

1.  **Clinical Presentation:** The patient is a 14-year-old boy with a 2-month histor...
✅ Processed Merge Q678: Answer = None
Processing row 679/1303...
⚠️ Could not extract answer from response for row 679:
Response: **Analysis of the Case:**

*   **Patient Profile:** 30s African American woman, no medical history.
...
✅ Processed Merge Q679: Answer = None
Processing row 680/1303...
⚠️ Could not extract answer from response for row 680:
Response:

⚠️ Could not extract answer from response for row 718:
Response: **Analysis:**

1.  **Clinical Presentation:** The patient presents with symptoms suggestive of acute...
✅ Processed Merge Q718: Answer = None
Processing row 719/1303...
⚠️ Could not extract answer from response for row 719:
Response: **Analysis:**

1.  **Identify the patient's key clinical features:**
    *   35-year-old female.
   ...
✅ Processed Merge Q719: Answer = None
Processing row 720/1303...
⚠️ Could not extract answer from response for row 720:
Response: **Analysis of the Case:**

*   **Patient:** 68-year-old male.
*   **Presentation:** 3-day history of...
✅ Processed Merge Q720: Answer = None
💾 Saved progress to CSV after 720 items
Processing row 721/1303...
⚠️ Could not extract answer from response for row 721:
Response: The patient presents with acute urinary incontinence and progressive bilateral weakness and tingling...
✅ Processed Merge Q721: Answer = None
Processing row 722/1303...
⚠️ Could not extract ans

✅ Processed Merge Q758: Answer = A
Processing row 759/1303...
⚠️ Could not extract answer from response for row 759:
Response: **Analysis of the Case:**

The patient is a 64-year-old male with a history of head and neck cancer ...
✅ Processed Merge Q759: Answer = None
Processing row 760/1303...
✅ Processed Merge Q760: Answer = A
💾 Saved progress to CSV after 760 items
Processing row 761/1303...
⚠️ Could not extract answer from response for row 761:
Response: **Analysis of the Case:**

1.  **Patient Presentation:** A 27-year-old woman with new onset hirsutis...
✅ Processed Merge Q761: Answer = None
Processing row 762/1303...
✅ Processed Merge Q762: Answer = C
Processing row 763/1303...
⚠️ Could not extract answer from response for row 763:
Response: **Analysis:**

1.  **Identify the condition:** The patient is a 4-day-old newborn presenting with bi...
✅ Processed Merge Q763: Answer = None
Processing row 764/1303...
✅ Processed Merge Q764: Answer = C
Processing row 765/1303...
⚠️ Could n

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv("[SR]_MedGemma27B_predictions_14B.csv")
df = pd.read_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_df3' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_df3' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_df3'] = df['data_source_df3']
else:
    print("Warning: 'data_source_df3' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_df3' in analysis_df.columns:
    category_stats = analysis_df.groupby('data_source_df3')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv("[SR]_MedGemma27B_predictions_14B.csv")
df = pd.read_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Count None/empty cells
def count_none_empty(series):
    """Count None, NaN, and empty string values"""
    return series.isna().sum() + (series == '').sum() + (series == 'None').sum()

# Overall None/empty counts
extracted_answer_none = count_none_empty(output_df['Extracted_Answer'])
answer_corr_none = count_none_empty(df['answer_corr'])

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print()
print(f"None/Empty in Extracted_Answer: {extracted_answer_none} ({extracted_answer_none/n*100:.2f}%)")
print(f"None/Empty in answer_corr: {answer_corr_none} ({answer_corr_none/n*100:.2f}%)")
print("=" * 60)
print()

# Analysis by category
if 'data_source_df3' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_df3' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_df3'] = df['data_source_df3']
else:
    print("Warning: 'data_source_df3' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_df3' in analysis_df.columns:
    # Add answer_corr to analysis_df for counting
    analysis_df['answer_corr'] = df['answer_corr']
    
    category_stats = analysis_df.groupby('data_source_df3').agg({
        'match': ['count', 'mean', 'std', lambda x: x.std() / np.sqrt(len(x))],
        'Extracted_Answer': lambda x: count_none_empty(x),
        'answer_corr': lambda x: count_none_empty(x)
    })
    
    # Flatten column names
    category_stats.columns = ['Count', 'Mean_Accuracy', 'Std_Dev', 'SE', 
                               'None_Empty_Extracted', 'None_Empty_Answer_Corr']
    category_stats = category_stats.reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    category_stats['None_Empty_Extracted_%'] = (category_stats['None_Empty_Extracted'] / category_stats['Count']) * 100
    category_stats['None_Empty_Answer_Corr_%'] = (category_stats['None_Empty_Answer_Corr'] / category_stats['Count']) * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 80)
    
    # Display main statistics
    display_cols = ['data_source_df3', 'Count', 'Mean_Accuracy_%', 'Std_Dev_%', 
                    'CI_95_Lower_%', 'CI_95_Upper_%']
    print(category_stats[display_cols].to_string(index=False))
    print()
    
    # Display None/Empty counts
    print("CATEGORY-WISE NONE/EMPTY COUNTS")
    print("-" * 80)
    none_cols = ['data_source_df3', 'Count', 'None_Empty_Extracted', 'None_Empty_Extracted_%',
                 'None_Empty_Answer_Corr', 'None_Empty_Answer_Corr_%']
    print(category_stats[none_cols].to_string(index=False))
    print("=" * 80)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 
               'Sample Size', 'None/Empty Extracted_Answer', 'None/Empty answer_corr'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n,
              f"{extracted_answer_none} ({extracted_answer_none/n*100:.2f}%)",
              f"{answer_corr_none} ({answer_corr_none/n*100:.2f}%)"]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)